# SQL Analysis - Stack Overflow Developer Survey (2023-2025)

## Objective
- Answer the main question using SQL queries against the cleaned dataset
- Same findings as 02 - Validated with a different tool

In [1]:
import pyodbc

# Connect to local SQL Server instance
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=so_survey_analysis;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
print("Connected successfully")

Connected successfully


## Load cleaned dataset in SQL Server

In [2]:
import pandas as pd
from sqlalchemy import create_engine

# Load dataset with outliers already cleaned
df = pd.read_csv("../data/processed/survey_analysis_ready.csv", low_memory=False)

# Connection details
server = "localhost\\SQLEXPRESS"
database = "so_survey_analysis"
driver = "ODBC+Driver+18+for+SQL+Server"

engine = create_engine(
    f"mssql+pyodbc://{server}/{database}?driver={driver}&trusted_connection=yes&TrustServerCertificate=yes"
)

# Push to SQL Server - Chunked (Error handling for a clean failure instead of a partial table)
try:
    df.to_sql("survey_ready", engine, if_exists="replace", index=False, chunksize=90, method="multi") # chunksize=90 - SQL Server's 2100 parameter limit
    print(f"Table loaded successfully - {len(df)} rows")
except Exception as e:
    print(f"Load failed: {e}")
    print("Table could be incomplete - Check before running queries")

Table loaded successfully - 203812 rows


### Query 1 - AISelect (% Using AI Tools)

In [3]:
# % using AI (any level), by role and year - same metric as pandas notebook
query = """
SELECT
    RoleClean,
    Year,
    SUM(CASE WHEN AISelect LIKE 'Yes%' THEN 1 ELSE 0 END) * 1.0 / COUNT(AISelect) AS pct_using_ai
FROM survey_ready
WHERE RoleClean IS NOT NULL
GROUP BY RoleClean, Year
ORDER BY RoleClean, Year
"""
result = pd.read_sql(query, engine)
print(result)

                 RoleClean  Year  pct_using_ai
0             Data Analyst  2023      0.416965
1             Data Analyst  2024      0.571705
2             Data Analyst  2025      0.696850
3            Data Engineer  2023      0.427885
4            Data Engineer  2024      0.607934
5            Data Engineer  2025      0.830252
6           Data Scientist  2023      0.600126
7           Data Scientist  2024      0.724725
8           Data Scientist  2025      0.824053
9   Database Administrator  2023      0.252918
10  Database Administrator  2024      0.427711
11  Database Administrator  2025      0.543478


#### Findings - AISelect

**Cross-check:** All 12 role/year percentages match with pandas notebook exactly

**Example:** Data Analyst 2025 (Pandas 69.7% - SQL 69.7%)

**Conclusion:** Same result, different tool - Validates the analysis

### Query 2 - AISent (% Favorable)

In [4]:
# % Favorable or very favorable toward AI by role and year - Same metric as pandas notebook
query = """
SELECT
    RoleClean,
    Year,
    SUM(CASE WHEN AISent IN ('Favorable', 'Very favorable') THEN 1 ELSE 0 END) * 1.0 / COUNT(AISent) AS pct_favorable
FROM survey_ready
WHERE RoleClean IS NOT NULL
GROUP BY RoleClean, Year
ORDER BY RoleClean, Year
"""
result = pd.read_sql(query, engine)
print(result)

                 RoleClean  Year  pct_favorable
0             Data Analyst  2023       0.803030
1             Data Analyst  2024       0.731458
2             Data Analyst  2025       0.632411
3            Data Engineer  2023       0.793333
4            Data Engineer  2024       0.752096
5            Data Engineer  2025       0.676271
6           Data Scientist  2023       0.824942
7           Data Scientist  2024       0.813478
8           Data Scientist  2025       0.661470
9   Database Administrator  2023       0.625767
10  Database Administrator  2024       0.672727
11  Database Administrator  2025       0.492647


#### Findings - AISent

**Cross-check:** All 12 percentages match pandas notebook exactly

**Conclusion:** Same result, different tool - Validates the analysis

### Query 3 - AIAcc (% Trust in Accuracy)

In [6]:
# % Who trust AI accuracy (Highly trust or Somewhat trust) by role and year
query = """
SELECT
    RoleClean,
    Year,
    SUM(CASE WHEN AIAcc IN ('Highly trust', 'Somewhat trust') THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(AIAcc), 0) AS pct_trust
FROM survey_ready
WHERE RoleClean IS NOT NULL
GROUP BY RoleClean, Year
ORDER BY RoleClean, Year
"""
result = pd.read_sql(query, engine)
print(result)

                 RoleClean  Year  pct_trust
0             Data Analyst  2023        NaN
1             Data Analyst  2024   0.552901
2             Data Analyst  2025   0.440476
3            Data Engineer  2023        NaN
4            Data Engineer  2024   0.453577
5            Data Engineer  2025   0.393888
6           Data Scientist  2023        NaN
7           Data Scientist  2024   0.506241
8           Data Scientist  2025   0.388393
9   Database Administrator  2023        NaN
10  Database Administrator  2024   0.614286
11  Database Administrator  2025   0.313433


**Note:**

- Query first returned `0.000000` for 2023 - No real data
- That number (not the NaN above) - Triggered the investigation below
- Nulling AIAcc in `01_data_cleaning.ipynb` - Re-running caused a divide by zero error (Fixed here with NULLIF)

### AIAcc Investigation (2023)

- Original `0.000000` did not look like "low trust" - Looked like no real data
- Checked what AIAcc actually has

#### Findings - AIAcc (2023)

**Found:** AIAcc has benefit style answers ("Greater efficiency") - Not trust ratings

**Conclusion:** Columns swapped in 2023 source file - Fixed in `01_data_cleaning.ipynb` (Also documented in `docs/schema_crosswalk.md`)

### Query 4 - AIComplex (% Good for Complex Tasks)

In [7]:
# % Who rate AI as good or very well for complex tasks by role and year
query = """
SELECT
    RoleClean,
    Year,
    SUM(CASE WHEN AIComplex IN ('Very well at handling complex tasks', 'Good, but not great at handling complex tasks') THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(AIComplex), 0) AS pct_good
FROM survey_ready
WHERE RoleClean IS NOT NULL
GROUP BY RoleClean, Year
ORDER BY RoleClean, Year
"""
result = pd.read_sql(query, engine)
print(result)

                 RoleClean  Year  pct_good
0             Data Analyst  2023       NaN
1             Data Analyst  2024  0.468966
2             Data Analyst  2025  0.351779
3            Data Engineer  2023       NaN
4            Data Engineer  2024  0.383792
5            Data Engineer  2025  0.321976
6           Data Scientist  2023       NaN
7           Data Scientist  2024  0.426982
8           Data Scientist  2025  0.395973
9   Database Administrator  2023       NaN
10  Database Administrator  2024  0.571429
11  Database Administrator  2025  0.238806


#### Findings - AIComplex

**Cross-check:** all 8 role/year percentages match pandas notebook exactly

**2023:** NaN as expected - AIComplex does not exist that year

**Conclusion:** Same result, different tool - Validates the analysis

### Query 5 - AIAgents (% Using AI Agents)

In [8]:
# % Using AI agents (daily, weekly, or monthly) by role - 2025 only
query = """
SELECT
    RoleClean,
    SUM(CASE WHEN AIAgents LIKE 'Yes%' AND AIAgents NOT LIKE '%copilot%' THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(AIAgents), 0) AS pct_using_agents
FROM survey_ready
WHERE RoleClean IS NOT NULL AND Year = 2025
GROUP BY RoleClean
ORDER BY RoleClean
"""
result = pd.read_sql(query, engine)
print(result)

                RoleClean  pct_using_agents
0            Data Analyst          0.312500
1           Data Engineer          0.298214
2          Data Scientist          0.312354
3  Database Administrator          0.222222


#### Findings - AIAgents

**Cross-check:** All 4 role percentages match pandas notebook exactly

**Conclusion:** Same result, different tool - Validates the analysis